## Setup

In [5]:
# Importing relevant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import warnings
warnings.filterwarnings("ignore")

# CHANGE DATA PATH HERE
data_path = "C:\\Users\\adams\\Desktop\\data\\CAR Data\\"

# CHANGE DATA PATH HERE, USE THE OUTPUT OF KRISH'S CAR DATA AGGREGATION FLOW
df_main = pd.read_csv(Path(data_path) / "df_main_cleaned.csv", parse_dates=['Activity Start Timestamp'])

df_last = (
    df_main.sort_values('Activity Start Timestamp')
           .groupby('Contact Session ID', as_index=False)
           .last()
)

## Verbiage Analysis 1: Voicemail vs. Not Voicemail

In [6]:
# OBJECTIVE: Determine, of all calls, how many lead to the clinic voicemail as opposed to anything other than a voicemail
# REASON: Gain additional information on whether it may be worthy to put option 4 (leads to a voicemail) before option 3 (does not lead to a voicemail)

# Get the final (latest) row for each Contact Session ID
df_last = (
    df_main.sort_values('Activity Start Timestamp')
           .groupby('Contact Session ID', as_index=False)
           .last()
)

# Check how many have missing Queue Name
missing_queue = df_last['Queue Name'].isna().sum()
print(f"Number of calls with missing final Queue Name: {missing_queue}")

# Exclude those missing values ---
df_last_valid = df_last.dropna(subset=['Queue Name'])

# Create flags for voicemail vs other ---
df_last_valid['Call Type'] = df_last_valid['Queue Name'].apply(
    lambda x: 'Voicemail' if x == 'Clinic Voicemail Transfer' else 'Other'
)

# Count totals and proportions ---
call_counts = df_last_valid['Call Type'].value_counts()
call_proportions = df_last_valid['Call Type'].value_counts(normalize=True) * 100

print("\n--- Call Counts ---")
print(call_counts)

print("\n--- Call Proportions (%) ---")
print(call_proportions.round(2))

# See top non-voicemail destinations ---
top_queues = (
    df_last_valid[df_last_valid['Call Type'] == 'Other']['Queue Name']
    .value_counts()
    .head(20)
)
print("\n--- Top 10 Non-Voicemail Queue Names ---")
print(top_queues)

Number of calls with missing final Queue Name: 103385

--- Call Counts ---
Other        51971
Voicemail    18174
Name: Call Type, dtype: int64

--- Call Proportions (%) ---
Other        74.09
Voicemail    25.91
Name: Call Type, dtype: float64

--- Top 10 Non-Voicemail Queue Names ---
Staff Directory English Transfer       19602
Front Desk Transfer                    12386
Intake Outdial Queue                    6753
Staff Directory Spanish Transfer        3000
Criminal Records Voicemail Transfer     1608
Family                                  1203
SubSenior Other                          827
Consumer                                 818
Housing                                  665
HIV Voicemail Transfer                   646
SubSenior Benefits                       527
Benefits                                 504
SubSenior Tenant                         490
SubSenior Consumer                       362
SubSenior Family                         328
Employment                              

In [7]:
# Investigate cause of all the unknown final queues

missing_queues_sessions = df_last[df_last['Queue Name'].isna()]['Contact Session ID']
termination_summary = (
    df_main[df_main['Contact Session ID'].isin(missing_queues_sessions)]
    ['Termination Reason']
    .value_counts()
)
print(termination_summary)

Customer Left                      9369
System disconnected the contact     741
NO_ANSWER_FROM_CUSTOMER              50
System Error                         28
Name: Termination Reason, dtype: int64


## Verbiage Analysis 2: Complaint/Compliment Menu

In [8]:
# OBJECTIVE: Determine how many calls reached compliment/complaint menu, and how many of these actually used an agent's time
# FIRST ATTEMPT: Flawed (look at following code for my final and better analysis)

# Filter to calls from March 16, 2025 onward
df_filtered = df_main[df_main['Activity Start Timestamp'] >= '2025-03-16']

# Find all calls that reached the Compliment or Complaint menu
complaint_calls = df_filtered[df_filtered['Activity Name'] == 'ComplimentOrComplaintMenu']

# Get unique Contact Session IDs for those calls
complaint_call_ids = complaint_calls['Contact Session ID'].unique()

# Subset full dataset to just those sessions
subset = df_filtered[df_filtered['Contact Session ID'].isin(complaint_call_ids)]

# Group by Contact Session ID and see which sessions ever had an Agent Name
grouped = subset.groupby('Contact Session ID')['Agent Name'].apply(lambda x: x.notna().any())

# Results
num_total = len(grouped)
num_with_agent = grouped.sum()
num_without_agent = num_total - num_with_agent

print(f"Total calls that reached Compliment/Complaint Menu: {num_total}")
print(f"Calls where an Agent Name appeared (spoke to agent): {num_with_agent}")
print(f"Calls where no Agent Name appeared (likely voicemail/no agent): {num_without_agent}")

# Optional: percentage
print(f"Percent that reached an agent: {num_with_agent / num_total * 100:.2f}%")

subset.head(15)

Total calls that reached Compliment/Complaint Menu: 63
Calls where an Agent Name appeared (spoke to agent): 28
Calls where no Agent Name appeared (likely voicemail/no agent): 35
Percent that reached an agent: 44.44%


,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason
2088449,d474178a-c0ac-475d-8b13-3340a5211c00,Main Number Telephony EP,NaN,NaN,2025-03-18 10:36:37,NaN,NaN,NaN
2088450,d474178a-c0ac-475d-8b13-3340a5211c00,NaN,LACMain,NaN,2025-03-18 10:36:37,NaN,NaN,NaN
2088451,d474178a-c0ac-475d-8b13-3340a5211c00,Main Number Telephony EP,NaN,LanguageSelectionMenu,2025-03-18 10:36:37,NaN,NaN,NaN
2088452,d474178a-c0ac-475d-8b13-3340a5211c00,Main Number Telephony EP,LACMain,NaN,2025-03-18 10:36:37,NaN,NaN,NaN
2088453,d474178a-c0ac-475d-8b13-3340a5211c00,Main Number Telephony EP,NaN,MainMenu,2025-03-18 10:36:45,NaN,NaN,NaN
2088465,d474178a-c0ac-475d-8b13-3340a5211c00,Main Number Telephony EP,NaN,ComplimentOrComplaintMenu,2025-03-18 10:37:41,NaN,NaN,NaN
2088466,d474178a-c0ac-475d-8b13-3340a5211c00,Main Number Telephony EP,NaN,MainMenu,2025-03-18 10:37:42,NaN,NaN,NaN
2088493,d474178a-c0ac-475d-8b13-3340a5211c00,Main Number Telephony EP,NaN,HelpWithLegalorOtherReasonMenu,2025-03-18 10:38:35,NaN,NaN,NaN
2088496,d474178a-c0ac-475d-8b13-3340a5211c00,Main Number Telephony EP,NaN,FrontDeskTransfer2,2025-03-18 10:38:44,NaN,NaN,NaN
2088497,d474178a-c0ac-475d-8b13-3340a5211c00,NaN,NaN,NaN,2025-03-18 10:38:44,Front Desk Transfer,NaN,NaN


In [9]:
import pandas as pd

# 1. Filter to March 16, 2025 onwards
df_filtered = df_main[df_main['Activity Start Timestamp'] >= '2025-03-16'].copy()

# Make sure timestamps are in datetime format
df_filtered['Activity Start Timestamp'] = pd.to_datetime(df_filtered['Activity Start Timestamp'])

# 2. Sort by session and timestamp to maintain order
df_filtered = df_filtered.sort_values(['Contact Session ID', 'Activity Start Timestamp'])

# 3. Find all Compliment/Complaint menu entries
complaint_entries = df_filtered[df_filtered['Activity Name'] == 'ComplimentOrComplaintMenu']

valid_sessions = []

# 4. For each such session, check if MainMenu appears within 60 seconds *after* Compliment/Complaint
for session_id, group in df_filtered.groupby('Contact Session ID'):
    complaint_times = group.loc[group['Activity Name'] == 'ComplimentOrComplaintMenu', 'Activity Start Timestamp']
    if complaint_times.empty:
        continue

    # For each time they entered the Compliment/Complaint menu
    backed_out = False
    for t in complaint_times:
        # Find any 'MainMenu' activity within 60 seconds *after* this timestamp
        back_to_main = group[
            (group['Activity Name'] == 'MainMenu') &
            (group['Activity Start Timestamp'] > t) &
            (group['Activity Start Timestamp'] <= t + pd.Timedelta(seconds=60))
        ]
        if not back_to_main.empty:
            backed_out = True
            break

    # Keep only those who did NOT go back to main within 1 minute
    if not backed_out:
        valid_sessions.append(session_id)

# 5. Subset to those sessions
subset = df_filtered[df_filtered['Contact Session ID'].isin(valid_sessions)]

# 6. Check whether they ever had an agent
grouped = subset.groupby('Contact Session ID')['Agent Name'].apply(lambda x: x.notna().any())

# 7. Summaries
num_total = len(grouped)
num_with_agent = grouped.sum()
num_without_agent = num_total - num_with_agent

print(f"Total sessions that stayed in Compliment/Complaint (did not return to Main Menu): {num_total}")
print(f"Sessions with an Agent Name (spoke to agent): {num_with_agent}")
print(f"Sessions without an Agent Name (likely voicemail): {num_without_agent}")
print(f"Percent that reached an agent: {num_with_agent / num_total * 100:.2f}%")

Total sessions that stayed in Compliment/Complaint (did not return to Main Menu): 17
Sessions with an Agent Name (spoke to agent): 15
Sessions without an Agent Name (likely voicemail): 2
Percent that reached an agent: 88.24%


In [10]:
# Create a dataframe of only the calls that passed the filter
survivor_data = df_filtered[df_filtered['Contact Session ID'].isin(valid_sessions)].copy()

# Sort by session and time for readability
survivor_data = survivor_data.sort_values(['Contact Session ID', 'Activity Start Timestamp'])

# Display the first few rows neatly
print("Sample of surviving compliment/complaint sessions:")
display_cols = [
    'Contact Session ID',
    'Activity Start Timestamp',
    'Activity Name',
    'Queue Name',
    'Agent Name',
    'Termination Reason'
]
survivor_data[display_cols].head(30)


Sample of surviving compliment/complaint sessions:


,Contact Session ID,Activity Start Timestamp,Activity Name,Queue Name,Agent Name,Termination Reason
2098846,00b976b5-8d07-4e0e-9fe8-dfb1ef8c0a68,2025-03-19 11:30:22,NaN,NaN,NaN,NaN
2098847,00b976b5-8d07-4e0e-9fe8-dfb1ef8c0a68,2025-03-19 11:30:22,NaN,NaN,NaN,NaN
2098848,00b976b5-8d07-4e0e-9fe8-dfb1ef8c0a68,2025-03-19 11:30:22,LanguageSelectionMenu,NaN,NaN,NaN
2098850,00b976b5-8d07-4e0e-9fe8-dfb1ef8c0a68,2025-03-19 11:30:22,NaN,NaN,NaN,NaN
2098851,00b976b5-8d07-4e0e-9fe8-dfb1ef8c0a68,2025-03-19 11:30:29,MainMenu,NaN,NaN,NaN
2098874,00b976b5-8d07-4e0e-9fe8-dfb1ef8c0a68,2025-03-19 11:31:22,ComplimentOrComplaintMenu,NaN,NaN,NaN
2098882,00b976b5-8d07-4e0e-9fe8-dfb1ef8c0a68,2025-03-19 11:31:45,FrontDeskTransfer2,NaN,NaN,NaN
2098883,00b976b5-8d07-4e0e-9fe8-dfb1ef8c0a68,2025-03-19 11:31:45,NaN,Front Desk Transfer,NaN,NaN
2098884,00b976b5-8d07-4e0e-9fe8-dfb1ef8c0a68,2025-03-19 11:31:45,NaN,NaN,NaN,NaN
2098885,00b976b5-8d07-4e0e-9fe8-dfb1ef8c0a68,2025-03-19 11:31:46,NaN,Front Desk Transfer,CBT Agent,NaN
